In [1]:
from trot.config import configure_once

configure_once()

In [2]:
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 8
U = 12
t = 1.0
n = L 
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = (L//2,L//2)
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)


System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Thu Oct  1 17:56:55 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:         1.02
[[[1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 

Overwritten attributes  get_hcore get_ovlp  of <class 'pyscf.scf.uhf.UHF'>


np.float64(-1.1597070476816338)

In [3]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
orbitals = mf.mo_coeff
nocc = L//2
trial_data = np.block([
    [orbitals[0][:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[1][:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.1597070477  2.000000e+02           0       0.0
[eql   20/100]   -1.2063800878  2.002611e+02           0       2.1
[eql   40/100]   -1.2320963894  2.003371e+02           0       4.1
[eql   60/100]   -1.2502892037  2.000307e+02           0       5.7
[eql   80/100]   -1.3159092230  2.001455e+02           0       7.3
[eql  100/100]   -1.6546258209  2.011148e+02           0       8.9

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.4652175939   1.353e-01     -1.4652175939  2.001350e+02           0      0.087      13.3
[blk  100/500]   -1.3814918158   7.475e-02     -1.2974411683  1.993614e+02           0      0.079      17.3
[blk  150/500]   -1.3531553495   5.311e-02     -1.2965301595  1.99916

In [4]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
# _,orbitals = np.linalg.eig(h1)
orbitals = mf.mo_coeff
_data = np.block([
    [ orbitals[0,:,:L//2],       orbitals[1,:,:L//2]],
    [ orbitals[0,:,:L//2],          -orbitals[1,:,:L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted',trial_ops)
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.1597070477  2.000000e+02           0       0.0
[eql   20/100]   -1.2063800878  2.002611e+02           0       1.3
[eql   40/100]   -1.2320963894  2.003371e+02           0       2.4
[eql   60/100]   -1.2502892037  2.000307e+02           0       3.2
[eql   80/100]   -1.3159092230  2.001455e+02           0       4.0
[eql  100/100]   -1.6546258209  2.011148e+02           0       4.8

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.4652175939   1.353e-01     -1.4652175939  2.001350e+02           0      0.048       7.2
[blk  100/500]   -1.3814918158   7.475e-02     -1.2974411683  1.993614e+02           0      0.039       9.1
[blk  150/500]   -1.3531553495   5.311e-02     -1.2965301595  1.99916

In [6]:
from trot.gmps.driver import make_mps_cpmc_ops
from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import make_mps_trial
from trot.prop.types import QmcParamsMps

Ca, Cb = mf.mo_coeff[0][:, :L//2], mf.mo_coeff[1][:, :L//2]
ghf = GhfTrial(jnp.array(np.block([[Ca, Cb], [Ca, -Cb]])))
tensors = sd_to_gmps(Ca, Cb, mode="maximal").tensors
R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
M = np.zeros((4, 4))
M[0, 0] = 1.0
M[1:3, 1:3] = R
M[3, 3] = -1
new_tensors = [np.einsum("ps,asb->apb", M, np.asarray(A, dtype=float)) for A in tensors]
trial = make_mps_trial(new_tensors,nelec=(L//2, L//2))
params = QmcParamsMps(n_walkers=200, n_blocks=500, n_eql_blocks=100, seed=2,
                      orbital_plan="maximal", walker_channel_chi=None)
ops = make_mps_cpmc_ops(ham, trial, sys, params)
run = run_qmc(sys=sys, params=params, ham_data=ham, trial_data=trial, trial_ops=ops.trial_ops,
              meas_ops=ops.meas_ops, prop_ops=ops.prop_ops,
              prop_ctx=ops.prop_ops.build_prop_ctx(ham, None, params), block_fn=blocks.block)


[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.7407111942  2.000000e+02           0       0.0


KeyboardInterrupt: 

## Two ways to use the spin-rotated MPS trial

The rotation above is single-site: `new_tensors[i] = M · tensors[i]` on the physical leg, so every bond keeps its
dimension. What it breaks is the separate conservation of N_up and N_dn: `M` mixes |up> and |dn>, so the bond indices
keep a definite N = N_up + N_dn but no definite S_z. Two ways to go on:

* **Way A**, what `make_mps_trial(new_tensors, nelec=...)` above does (and production, `mps_cpmc_gpu` with
  `Config.trial_rotation`): project the rotated state exactly onto the walkers' (N_up, N_dn) sector
  (`trot.trial.mps.project_to_sector`). To have exact (N_up, N_dn) labels again, every bond index is split by the S_z
  its left block carries; then the result is compressed sector by sector.
* **Way B**: keep the rotated tensors as they are, labelled by N only.

The cells below compare the two: block structure, what the walkers see, and the contraction work.

In [ ]:
# The two representations of the same rotated trial. Needs only the UHF cell (mf, L, h1, U): the rotation of the
# cell above is rebuilt here, so its CPMC run does not have to be executed first.
from collections import Counter

from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import make_mps_trial, rotate_spin

nelec = (L // 2, L // 2)
Ca, Cb = mf.mo_coeff[0][:, :L // 2], mf.mo_coeff[1][:, :L // 2]
R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
M = np.zeros((4, 4))
M[0, 0], M[1:3, 1:3], M[3, 3] = 1.0, R, -1.0

gmps = sd_to_gmps(Ca, Cb, mode="maximal")  # the unrotated UHF determinant as an exact MPS, (N_up, N_dn) labels
way_b = rotate_spin(gmps.tensors, R)  # the single-site rotation only
assert all(np.allclose(x, np.einsum("ps,asb->apb", M, np.asarray(A))) for x, A in zip(way_b, gmps.tensors))
labels_b = [np.asarray(q).reshape(-1, 2).sum(axis=1) for q in gmps.charges]  # the rotation keeps N on every index
way_a = make_mps_trial(way_b, nelec=nelec)  # + the exact projection onto (N_up, N_dn): = `trial` in the cell above
labels_a = [np.asarray(q).reshape(-1, 2) for q in way_a.charges]

print(f"the projection keeps {way_a.sector_weight:.6f} of the rotated state's norm")
print("bond | way B: indices N-sectors largest | way A: indices (N_up,N_dn)-sectors largest | left 2S_z (way A)")
for b in range(L + 1):
    sectors_b = Counter(labels_b[b].tolist())
    sectors_a = Counter(map(tuple, labels_a[b].tolist()))
    sz = labels_a[b][:, 0] - labels_a[b][:, 1]
    print(f"{b:4d} | {len(labels_b[b]):14d} {len(sectors_b):9d} {max(sectors_b.values()):7d} | "
          f"{len(labels_a[b]):14d} {len(sectors_a):19d} {max(sectors_a.values()):7d} | {sz.min():+d} .. {sz.max():+d}")

In [ ]:
# What the walkers see. CPMC walkers have N_up = N_dn, i.e. S_z = 0, so they have no overlap with the S_z != 0 part
# of the rotated state: <A|phi> and <B|phi> differ by one constant (A is normalised after projection) and the local
# energies <T|H|phi>/<T|phi> are identical (H conserves S_z). Dense contractions, so no labels are needed here.
from trot.meas.mps import apply_mpo, hubbard_mpo_from_h1


def contract(bra, ket):
    """<bra|ket> of two real d = 4 MPS in the same local basis."""
    env = np.ones((1, 1))
    for x, y in zip(bra, ket):
        env = np.einsum("ab,apc,bpd->cd", env, np.asarray(x), np.asarray(y), optimize=True)
    return float(env[0, 0])


W = hubbard_mpo_from_h1(h1, U)
a_tensors = [np.asarray(x) for x in way_a.tensors]
h_a, h_b = apply_mpo(W, a_tensors), apply_mpo(W, way_b)

rng = np.random.default_rng(0)
print("walker |  <A|phi> / <B|phi>  |   E_L with B        E_L with A")
for k in range(6):  # random S_z = 0 determinants as exact MPS in the same local basis
    Wa = np.linalg.qr(rng.standard_normal((L, nelec[0])))[0]
    Wb = np.linalg.qr(rng.standard_normal((L, nelec[1])))[0]
    phi = sd_to_gmps(Wa, Wb, mode="maximal").tensors
    o_a, o_b = contract(a_tensors, phi), contract(way_b, phi)
    print(f"{k:6d} | {o_a / o_b:19.12f} | {contract(h_b, phi) / o_b:16.10f} {contract(h_a, phi) / o_a:16.10f}")
print(f"1/sqrt(sector weight) = {1 / np.sqrt(way_a.sector_weight):.12f}")

energy = lambda T, HT: contract(HT, T) / contract(T, T)
print(f"<T|H|T>: way B {energy(way_b, h_b):.10f} (the unrotated UHF energy, {mf.e_tot:.10f}: R commutes with H); "
      f"way A {energy(a_tensors, h_a):.10f} (the S_z = 0 part, what the walkers sample)")

In [ ]:
# Contraction work of <trial|walker> when trial and walker are contracted block by block: per bond, every walker
# sector (N_up, N_dn) times the trial sector it pairs with. Way A pairs it with the trial sector of the same
# (N_up, N_dn); way B (N labels only) with the whole trial sector of the same N; without labels everything pairs.
walker_labels = [np.asarray(q).reshape(-1, 2) for q in sd_to_gmps(Wa, Wb, mode="maximal").charges]
work = {"way A: (N_up, N_dn) blocks": 0, "way B: N blocks": 0, "no blocks": 0}
for b in range(L + 1):
    w = Counter(map(tuple, walker_labels[b].tolist()))
    sectors_a = Counter(map(tuple, labels_a[b].tolist()))
    sectors_b = Counter(labels_b[b].tolist())
    work["way A: (N_up, N_dn) blocks"] += sum(n * sectors_a.get(s, 0) for s, n in w.items())
    work["way B: N blocks"] += sum(n * sectors_b.get(s[0] + s[1], 0) for s, n in w.items())
    work["no blocks"] += len(walker_labels[b]) * len(labels_b[b])
for name, entries in work.items():
    print(f"{name:28s} {entries:8d} environment entries (summed over bonds)")

### Implications

| | **Way A**: rotate + project (`make_mps_trial`) | **Way B**: rotate only |
|---|---|---|
| state | the S_z = 0 part of the rotated state, normalised | the rotated state |
| bond labels | exact (N_up, N_dn): every charge-blocked contraction (trot's reference engine and the batched GPU engine) runs unchanged | only N: the (N_up, N_dn)-blocked code cannot use it; it would need an N-only pairing |
| bond dimension | more indices (each one split by its left-block S_z), but **no sector larger** than way B's N-sectors | unchanged |
| overlaps with S_z = 0 walkers | way B's times one constant (1/sqrt(sector weight)) | — |
| local energies | **identical** to way B | — |
| CPMC (ratios, constraint, mixed estimator) | **identical** to way B | — |
| ⟨T\|H\|T⟩ | the spin-filtered energy (what the walkers actually sample) | the unrotated energy (the rotation commutes with H) |
| contraction work | each walker sector pairs with its own (N_up, N_dn) sector: **≤ way B** | each walker sector pairs with a whole N-sector |

So the projection adds no information and no cost beyond a one-off host step: it is the rotated state with finer labels.
At L = 100 (DMRG trial, χ_T = 8, β = 90°) the middle bond goes from 8 indices in 3 N-sectors (largest 4) to 160 indices in
61 (N_up, N_dn) sectors (largest still 4), because the left half's 2S_z ranges from −20 to +20 after the rotation; with
χ_T = 16, 16 → 218 indices in 43 sectors, largest 8 in both. Production (`mps_cpmc_gpu`, `Config.trial_rotation`) uses way A.